 # Neuro-simboličko zaključivanje bolesti korišćenjem vektorske reprezentacije simptoma i graf baze

Ovaj notebook demonstrira neuro-simbolički pipeline za rangiranje potencijalnih bolesti na osnovu simptoma koje navodi korisnik. Fokus eksperimenta je na semantičkom mapiranju simptoma korišćenjem vektorskih reprezentacija i simboličkom zaključivanju nad medicinskom ontologijom smeštenom u graf bazi podataka Neo4j.

## Pretpostavka

U ovom eksperimentu pretpostavlja se postojanje prethodnog NLP sloja koji je već izdvojio relevantne simptome iz korisničkog unosa u prirodnom jeziku. Zbog toga se **ne razmatra proces ekstrakcije simptoma iz sirovog teksta**. Umesto toga, cilj je evaluacija sledećih koraka:

- Semantičko mapiranje izdvojenih simptoma na ontološke pojmove korišćenjem vektorske reprezentacije,

- Simboličko zaključivanje bolesti korišćenjem znanja iz Neo4j ontološkog grafa i.

In [ ]:
#extracted_symptoms = ["Dry cough", "fever", "fatigue", "headache", "sore throat", "shortness of breath"]
#extracted_symptoms = ["jaundice", "black urine", "Lethargy"]
#extracted_symptoms = ["DiarRhea", "Lethargy", "vomiting"]
#extracted_symptoms = ["diarrhea", "headache", "abdominal pain", "pharynx inflammation", "maculopapular rash", "bleeding", "nausea", "chills", "chest pain"]
#extracted_symptoms = ["lethargy", "cough", "fever", "nausea", "rhinorrhea"]
#extracted_symptoms = ["diarrhea", "headache", "rash", "bloodshot eye", "bleeding", "weakness", "muscle weakness", "fever"]
#extracted_symptoms = ["headache", "fever", "confusion", "disorientation", "stiff neck", "tremor", "coma"]
extracted_symptoms = ['fever', 'shivering', 'myalgia', 'headache']


## Vektorska reprezentacija

Svaki izdvojeni simptom mapira se u kontinuirani vektorski prostor korišćenjem unapred istreniranog **intfloat/multilingual-e5-large** modela. Ova faza omogućava semantičko poređenje između simptoma unetih od strane
korisnika i simptoma definisanih u ontologiji.

Vektorske reprezentacije za ontološke simptome generisani su pripremnoj fazi i trajno skladišteni, dok se vektorske reprezentacije za ulazne simptome računaju prilikom svakog korisničkog unosa.

In [ ]:
from sentence_transformers import SentenceTransformer

#model = SentenceTransformer('all-MiniLM-L6-v2')
#model = SentenceTransformer('NeuML/pubmedbert-base-embeddings')
model = SentenceTransformer('intfloat/multilingual-e5-large')

query_embeddings = model.encode(extracted_symptoms)

## Učitavanje ontoloških simbola i vektorskih reprezentacija

In [ ]:
import os
from neo4j import GraphDatabase
from dotenv import load_dotenv

load_dotenv(override=True)

neo4j_url = os.getenv("NEO4J_URL")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(neo4j_url, auth=(neo4j_username, neo4j_password))

In [ ]:
with driver.session() as session:
    result = session.run("""
        MATCH (s:Symptom)
        RETURN  s.n4sch__label[0] AS label, s.embedding AS embedding
    """)
    ontology_symptoms = [(r["label"], r["embedding"]) for r in result]

## Semantičko mapiranje simptoma

Za poređenje vektorskih reprezentacija ulaznih simptoma i simptoma iz ontologije skladištenih u graf bazi podataka Neo4j koristi se mera kosinusne sličnosti.
Za svaki ulazni simptom identifikuje se ontološki simptom sa najvećom vrednošću sličnosti, koji se zatim koristi kao njegov semantički ekvivalent u daljem procesu zaključivanja.

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

onto_labels = [o[0] for o in ontology_symptoms]
onto_vectors = np.array([o[1] for o in ontology_symptoms])

matched = []

for symptom, q_emb in zip(extracted_symptoms, query_embeddings):
    sims = cosine_similarity([q_emb], onto_vectors)[0]
    best_idx = sims.argmax()

    matched.append({
        "input_symptom": symptom,
        "mapped_symptom": onto_labels[best_idx],
        "confidence": float(sims[best_idx])
    })

In [ ]:
CONFIDENCE_THRESHOLD = 0.9

print(f"{'USER INPUT':<25} | {'ONTOLOGICAL TERM':<30} | {'SIMILARITY (0-1)':<10}")
print("-" * 75)

mapped_symptoms_list = []

for m in matched:
    if m["confidence"] > CONFIDENCE_THRESHOLD:
        print(f"{m['input_symptom']:<25} | {m['mapped_symptom']:<30} | {m['confidence']:.4f}")
        mapped_symptoms_list.append(m['mapped_symptom'])
    else:
        print(f"{m['input_symptom']:<25} | {m['mapped_symptom']:<30} | {m['confidence']:.4f} -- UNDER THRESHOLD")

print(f"\nFinal List {mapped_symptoms_list}")

## Neo4j zaključivanje

Prvo se pripremaju prepoznati semantički ekvivalenti simptoma. Biraju se simptomi sa **confidence score**-om većim od 0.9.

In [ ]:
mapped_symptoms = [m["mapped_symptom"] for m in matched if m["confidence"] > CONFIDENCE_THRESHOLD]

mapped_symptoms

Rangiranje bolesti se ne vrši jednostavnim brojanjem poklapanja simptoma, već se koristi ponderisani zbir simptoma prisutnih kod bolesti. Svaki simptom ima pridruženu težinu (weight) koja je prethodno izračunata na osnovu **Information Content (IC)**, što odražava koliko je simptom informativan za diferencijaciju bolesti.

Upit u Neo4j graf bazi sumira ove težine za sve simptome bolesti koje odgovaraju ulaznim simptomima korisnika (total_score). Da bi se rangiranje prilagodilo i za bolesti sa različitim brojem simptoma, ukupni skor se normalizuje deljenjem sa kvadratnim korenom broja ukupnih simptoma bolesti (normalized_score). Tako se postiže balans između:

1. Specifičnosti simptoma (simptomi sa većim IC imaju veći doprinos),

2. Broja simptoma bolesti (da bolesti sa mnogo simptoma ne dominiraju).

Ovaj pristup omogućava da bolesti sa manjim brojem, ali informativnijih simptoma mogu biti rangirane više od bolesti sa mnogo, ali manje specifičnih simptoma.

In [ ]:
MIN_MATCH   = 2
TOP_K       = 5
HAS_SYMPTOM = "http://purl.obolibrary.org/obo/RO_0002452"

from pathlib import Path

BASE_DIR = Path.cwd()

CYPHER_QUERY_PATH = BASE_DIR / "tests" / "inference-and-scoring-test" / "queries" / "cypher-query-2.cyp"

with open(CYPHER_QUERY_PATH) as f:
    raw = f.read()

QUERY = raw
print(f"Uspešno učitan Cypher Query.")

In [ ]:
total_input_symptoms = len(mapped_symptoms)

with driver.session() as session:
    r = session.run(QUERY, has_symptom=HAS_SYMPTOM, symptoms=mapped_symptoms, min_match=MIN_MATCH)

    records = list(r)

    # Header sa novim poljem
    print(f"{'DISEASE':<40} | {'URI':<50} | {'SCORE':<6} | {'MATCH':<5} | {'D.COV':<7} | {'I.COV':<7} | {'MATCHED'} | {'MISSING'}")
    print("-" * 220)

    for rec in records:
        disease_cov = round(rec['match_count'] / rec['total_symptom_count'] * 100, 1)
        input_cov   = round(rec['match_count'] / total_input_symptoms * 100, 1)
        print(f"{rec['disease']:<40} | "
              f"{rec['uri']:<50} | "
              f"{rec['normalized_score']:.2f}   | "
              f"{rec['match_count']:<5} | "
              f"{disease_cov}%{'':<2} | "
              f"{input_cov}%{'':<2} | "
              f"{rec['matched_symptoms']} | "
              f"{rec['missing_symptoms']} | ")

## Testiranje modela za vektorsku reprezentaciju

Rezultati i metodologija testiranja je detaljno opisana u **embedding-test.docx**.

In [ ]:
from pathlib import Path
import json

BASE_DIR = Path.cwd()

test_cases_path = BASE_DIR / "tests" / "embeddings-test" / "symptom-full-test.json"

with open(test_cases_path, "r", encoding="utf-8") as f:
    test_cases = json.load(f)

print(f"Učitano {len(test_cases)} test slučajeva")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import numpy as np

#model = SentenceTransformer('all-MiniLM-L6-v2')
#model = SentenceTransformer('NeuML/pubmedbert-base-embeddings')
model = SentenceTransformer('intfloat/multilingual-e5-large')

all_embeddings = []
all_metadata = []
matched = {}
for case in test_cases:
    symptoms = case.get("extracted_symptoms", [])
    case_id = case.get("id", "unknown")

    if not symptoms:
        continue

    if case_id not in matched:
        matched[case_id] = []

    for symptom in symptoms:
        embedding = model.encode(symptom.strip(), normalize_embeddings=True)
        onto_labels = [o[0] for o in ontology_symptoms]
        onto_vectors = np.array([o[1] for o in ontology_symptoms])

        sims = cosine_similarity([embedding], onto_vectors)[0]
        best_idx = sims.argmax()

        matched[case_id].append({
            "input_symptom": symptom,
            "mapped_symptom": onto_labels[best_idx],
            "confidence": float(sims[best_idx])
        })

In [ ]:
def evaluate_embedding_layer(matched_results):
    usable_threshold = 0.90

    def empty_stats():
        return {
            "total": 0,
            "exact": 0,
            "usable": 0,
            "fail": 0,
            "conf_sum": 0.0,
            "fail_cases": [],
            "non_exact_usable": []
        }

    categories = {
        "EXACT":  empty_stats(),
        "SYN":    empty_stats(),
        "LLM":   empty_stats(),
        "SUM": empty_stats(),
    }

    for case_id, mappings in matched_results.items():
        if not mappings:
            continue

        best = mappings[0]
        conf      = best["confidence"]
        input_sym = best["input_symptom"]
        mapped_sym = best["mapped_symptom"]
        is_exact  = (input_sym.strip().lower() == mapped_sym.strip().lower())

        if case_id.startswith("EXACT"):
            cat = "EXACT"
        elif case_id.startswith("SYN"):
            cat = "SYN"
        elif case_id.startswith("LLM"):
            cat = "LLM"
        else:
            cat = "SUM"

        for key in [cat, "SUM"]:
            s = categories[key]
            s["total"]    += 1
            s["conf_sum"] += conf

            if is_exact:
                s["exact"] += 1

            if conf >= usable_threshold:
                s["usable"] += 1
                if not is_exact:
                    s["non_exact_usable"].append((case_id, input_sym, mapped_sym, conf))
            else:
                s["fail"] += 1
                s["fail_cases"].append((case_id, input_sym, mapped_sym, conf))

    def print_category(name, s):
        total = s["total"]
        if total == 0:
            print(f"\n[{name}] — Nema podataka")
            return

        exact_rate   = (s["exact"]  / total) * 100
        usable_rate  = (s["usable"] / total) * 100
        bad_rate     = (s["fail"]   / total) * 100
        avg_conf     = s["conf_sum"] / total

        is_summary = (name == "SUM")
        sep = "═" * 60 if is_summary else "─" * 60

        print(f"\n{sep}")
        if is_summary:
            print(f"{'  ZBIRNA ANALIZA — SVE KATEGORIJE  ':^60}")
        else:
            desc = {"EXACT": "180 termina iz ontologije", "SYN": "70 sinonima", "LLM": "llm case-ovi"}.get(name, "")
            print(f"{'  KATEGORIJA: ' + name + ' (' + str(total) + ' testova' + ((' — ' + desc) if desc else '') + ')  ':^60}")
        print(sep)

        print(f"  {'Metrika':<32} | {'Vrednost':>9} | Status")
        print(f"  {'─'*32}─+─{'─'*9}─+─{'─'*6}")

        print(f"  {'Exact Match Rate':<32} | {exact_rate:>8.1f}%")
        print(f"  {'Usable Match Rate  (conf ≥ 0.90)':<32} | {usable_rate:>8.1f}%")
        print(f"  {'Average Confidence':<32} | {avg_conf:>9.4f}")
        print(f"  {'Bad Match Rate     (conf < 0.90)':<32} | {bad_rate:>8.1f}%")
        print(f"  {'─'*55}")
        print(f"  Exact: {s['exact']}  |  Usable (non-exact): {len(s['non_exact_usable'])}  |  Fail: {s['fail']}  |  Total: {total}")

    for cat in ["EXACT", "SYN", "LLM"]:
        print_category(cat, categories[cat])

    print_category("SUM", categories["SUM"])

    return categories

In [ ]:
results = evaluate_embedding_layer(matched)

# Testiranje procesa zakljucivanje (Inference)

Rezultati i metodologija testiranja je detaljno opisana u **inference-test.docx**.

## Cypher upit

In [ ]:
MIN_MATCH   = 2
TOP_K       = 5
HAS_SYMPTOM = "http://purl.obolibrary.org/obo/RO_0002452"

from pathlib import Path

BASE_DIR = Path.cwd()

CYPHER_QUERY_PATH = BASE_DIR / "tests" / "inference-and-scoring-test" / "queries" / "cypher-query-2.cyp"

with open(CYPHER_QUERY_PATH) as f:
    raw = f.read()

QUERY = raw
print(f"Uspešno učitan Cypher Query.")

## Test podaci (Ground Truth)

In [ ]:
from pathlib import Path
import json

BASE_DIR = Path.cwd()

GROUND_TRUTH_PATH = BASE_DIR / "tests" / "inference-and-scoring-test" / "symptom-disease-test-data.json"

with open(GROUND_TRUTH_PATH) as f:
    raw = json.load(f)

ground_truth = [
    {
        "disease":  item["disease"][0],
        "symptoms": [s[0] for s in item["symptoms"]]
    }
    for item in raw
    if len(item["symptoms"]) >= MIN_MATCH
]

print(f"Učitano {len(ground_truth)} test slučajeva (bolesti sa >= {MIN_MATCH} simptoma)")

In [ ]:
def run_query(session, test_symptoms):
    query_result = session.run(
        QUERY,
        has_symptom=HAS_SYMPTOM,
        symptoms=test_symptoms,
        min_match=MIN_MATCH
    )
    return list(query_result)


def evaluate(ground_truth, session, drop_n=0):
    hit1 = hit3 = hit5 = total = no_results = 0
    details = []

    for item in ground_truth:
        disease  = item["disease"]
        symptoms = item["symptoms"]

        if len(symptoms) <= drop_n:
            continue

        input_symptoms = symptoms[:-drop_n] if drop_n > 0 else symptoms
        records = run_query(session, input_symptoms)
        ranked  = [r["disease"] for r in records]
        total  += 1

        if not ranked:
            no_results += 1
            details.append({"disease": disease, "symptoms": input_symptoms,
                            "ranked": [], "rank": None,
                            "hit@1": False, "hit@3": False, "hit@5": False})
            continue

        rank = ranked.index(disease) + 1 if disease in ranked else None
        h1 = rank == 1     if rank else False
        h3 = rank <= 3     if rank else False
        h5 = rank <= TOP_K if rank else False

        if h1: hit1 += 1
        if h3: hit3 += 1
        if h5: hit5 += 1

        details.append({
            "disease":   disease,
            "symptoms":  input_symptoms,
            "ranked":    ranked[:TOP_K],
            "rank":      rank,
            "hit@1":     h1,
            "hit@3":     h3,
            "hit@5":     h5,
            "top_score": records[0]["total_score"] if records else None
        })

    return {
        "hit@1": hit1, "hit@3": hit3, "hit@5": hit5,
        "total": total, "no_results": no_results,
        "details": details
    }

In [ ]:
with driver.session() as session:
    full = evaluate(ground_truth, session, drop_n=0)

t = full["total"]
print(f"{'='*50}")
print(f"FULL MATCH (svi simptomi)")
print(f"{'='*50}")
print(f"Total test cases : {t}")
print(f"No results       : {full['no_results']}")
print(f"Hit@1            : {full['hit@1']}/{t}  ({full['hit@1']/t*100:.1f}%)")
print(f"Hit@3            : {full['hit@3']}/{t}  ({full['hit@3']/t*100:.1f}%)")
print(f"Hit@5            : {full['hit@5']}/{t}  ({full['hit@5']/t*100:.1f}%)")

#print(full["details"])

In [ ]:
with driver.session() as session:
    partial1 = evaluate(ground_truth, session, drop_n=1)
    partial2 = evaluate(ground_truth, session, drop_n=2)

#print(partial1["details"])
#print(partial2["details"])

for label, res in [("PARTIAL (drop=1)", partial1), ("PARTIAL (drop=2)", partial2)]:
    t = res["total"]
    print(f"{'='*50}")
    print(f"{label}")
    print(f"{'='*50}")
    print(f"Total : {t}")
    print(f"Hit@1 : {res['hit@1']}/{t}  ({res['hit@1']/t*100:.1f}%)")
    print(f"Hit@3 : {res['hit@3']}/{t}  ({res['hit@3']/t*100:.1f}%)")
    print(f"Hit@5 : {res['hit@5']}/{t}  ({res['hit@5']/t*100:.1f}%)")
    print()

In [ ]:
misses = [d for d in full["details"] if not d["hit@5"]]
print(f"Potpuni promašaji (nije u top-{TOP_K}): {len(misses)}\n")

for m in misses:
    print(f"  Bolest   : {m['disease']}")
    print(f"  Simptomi : {m['symptoms']}")
    print(f"  Top-5    : {m['ranked']}")
    print()